# Advanced Pandas: Combining Data
v.ekc

Real datasets rarely arrive as one table. Today we learn three ways to combine DataFrames: `merge`, `join`, and `concat` — and when each one is the right tool.

| Section | Topic |
|---------|-------|
| 1 | Setup |
| 2 | Merge |
| 3 | Join |
| 4 | Concatenate |
| 5 | 🔬 Activity — Humboldt Student Data |
| Appendix | Quick Reference |

---
## 1. Setup

In [94]:
import numpy as np
import pandas as pd

---
## 2. Merge

We'll use two DataFrames of Gilmore Girls characters as a toy example.



| Merge type | Code | What rows are kept |
|------------|------|--------------------|
| Inner (default) | `df1.merge(df2)` | Only rows matching in both |
| Outer | `df1.merge(df2, how='outer')` | All rows from both |
| Left | `df1.merge(df2, how='left')` | All rows from left df |
| Right | `df1.merge(df2, how='right')` | All rows from right df |
| Different col names | `df1.merge(df2, left_on='a', right_on='b')` | Specify each key column |
| Merge on index | `df1.merge(df2, left_on='col', right_index=True)` | Right df is indexed by key |
| Debug | `df1.merge(df2, how='outer', indicator=True)` | Adds `_merge` column showing origin |

In [95]:
# Create the first dataframe
df1 = pd.DataFrame({
    'Name': ['Lorelai', 'Rory', 'Luke', 'Emily'],
    'Occupation': ['Manager', 'Student', 'Owner', 'Socialite'],
    'Age': [32, 20, 40, 60]
})

# Create the second dataframe
df2 = pd.DataFrame({
    'Name': ['Lorelai', 'Rory', 'Sookie', 'Richard'],
    'Home': ['Stars Hollow', 'Stars Hollow', 'Stars Hollow', 'Hartford'],
    'Age': [32, 20, 12, 70]
})

In [96]:
df1

,Name,Occupation,Age
0,Lorelai,Manager,32
1,Rory,Student,20
2,Luke,Owner,40
3,Emily,Socialite,60


In [97]:
df2

,Name,Home,Age
0,Lorelai,Stars Hollow,32
1,Rory,Stars Hollow,20
2,Sookie,Stars Hollow,12
3,Richard,Hartford,70


In [98]:
# A standard merge (inner)
df1.merge(df2)

,Name,Occupation,Age,Home
0,Lorelai,Manager,32,Stars Hollow
1,Rory,Student,20,Stars Hollow


In [99]:
# Explicitly specifying what to merge by (same as before)
df1.merge(df2, on = 'Name')

,Name,Occupation,Age_x,Home,Age_y
0,Lorelai,Manager,32,Stars Hollow,32
1,Rory,Student,20,Stars Hollow,20


In [100]:
# What if they had different column names?
df2.rename(columns = {'Name':'Character Name'},inplace=True)
df2

,Character Name,Home,Age
0,Lorelai,Stars Hollow,32
1,Rory,Stars Hollow,20
2,Sookie,Stars Hollow,12
3,Richard,Hartford,70


In [101]:
df1.columns

Index(['Name', 'Occupation', 'Age'], dtype='str')

In [102]:
df2.columns

Index(['Character Name', 'Home', 'Age'], dtype='str')

In [103]:
# Try merging now (error!)
df1.merge(df2)


# Specify one of the column names (error!)
#    obviously this will not work either!!! no shared column name :(
# df1.merge(df2, on = 'Name')

,Name,Occupation,Age,Character Name,Home
0,Lorelai,Manager,32,Lorelai,Stars Hollow
1,Rory,Student,20,Rory,Stars Hollow


In [104]:
# How to specify both names if they are different
df1.merge(df2, left_on = 'Name',right_on = 'Character Name')

,Name,Occupation,Age_x,Character Name,Home,Age_y
0,Lorelai,Manager,32,Lorelai,Stars Hollow,32
1,Rory,Student,20,Rory,Stars Hollow,20


In [105]:
# Can drop the redundant column
df1.merge(df2, left_on = 'Name',right_on = 'Character Name').drop(columns = 'Character Name')

,Name,Occupation,Age_x,Home,Age_y
0,Lorelai,Manager,32,Stars Hollow,32
1,Rory,Student,20,Stars Hollow,20


In [106]:
# Reset it back to original
df2.rename(columns = {'Character Name':'Name'},inplace=True)

In [107]:
# Outer
df1.merge(df2, how = 'outer')

,Name,Occupation,Age,Home
0,Emily,Socialite,60,NaN
1,Lorelai,Manager,32,Stars Hollow
2,Luke,Owner,40,NaN
3,Richard,NaN,70,Hartford
4,Rory,Student,20,Stars Hollow
5,Sookie,NaN,12,Stars Hollow


In [108]:
# Use the indicator parameter to keep track of where the rows came from (good for debugging)
df1.merge(df2, how = 'outer',indicator=True)

,Name,Occupation,Age,Home,_merge
0,Emily,Socialite,60,NaN,left_only
1,Lorelai,Manager,32,Stars Hollow,both
2,Luke,Owner,40,NaN,left_only
3,Richard,NaN,70,Hartford,right_only
4,Rory,Student,20,Stars Hollow,both
5,Sookie,NaN,12,Stars Hollow,right_only


Reset!

In [109]:
df1

,Name,Occupation,Age
0,Lorelai,Manager,32
1,Rory,Student,20
2,Luke,Owner,40
3,Emily,Socialite,60


In [110]:
df2

,Name,Home,Age
0,Lorelai,Stars Hollow,32
1,Rory,Stars Hollow,20
2,Sookie,Stars Hollow,12
3,Richard,Hartford,70


In [111]:
# Left
df1.merge(df2,how = 'left')

,Name,Occupation,Age,Home
0,Lorelai,Manager,32,Stars Hollow
1,Rory,Student,20,Stars Hollow
2,Luke,Owner,40,NaN
3,Emily,Socialite,60,NaN


In [112]:
# Right
df1.merge(df2,how = 'right')

,Name,Occupation,Age,Home
0,Lorelai,Manager,32,Stars Hollow
1,Rory,Student,20,Stars Hollow
2,Sookie,NaN,12,Stars Hollow
3,Richard,NaN,70,Hartford


In [113]:
# Cross join (less common, but occasionally handy)
df1.merge(df2,how='cross')

,Name_x,Occupation,Age_x,Name_y,Home,Age_y
0,Lorelai,Manager,32,Lorelai,Stars Hollow,32
1,Lorelai,Manager,32,Rory,Stars Hollow,20
2,Lorelai,Manager,32,Sookie,Stars Hollow,12
3,Lorelai,Manager,32,Richard,Hartford,70
4,Rory,Student,20,Lorelai,Stars Hollow,32
5,Rory,Student,20,Rory,Stars Hollow,20
6,Rory,Student,20,Sookie,Stars Hollow,12
7,Rory,Student,20,Richard,Hartford,70
8,Luke,Owner,40,Lorelai,Stars Hollow,32
9,Luke,Owner,40,Rory,Stars Hollow,20


In [114]:
# What if there are two common columns?
df1['School'] = ['Hartford Community College','Yale','Stars Hollow High','Smith College']
df2['School'] = ['Hartford Community College','Yale','Unknown','Yale']

In [115]:
df1

,Name,Occupation,Age,School
0,Lorelai,Manager,32,Hartford Community College
1,Rory,Student,20,Yale
2,Luke,Owner,40,Stars Hollow High
3,Emily,Socialite,60,Smith College


In [116]:
df2

,Name,Home,Age,School
0,Lorelai,Stars Hollow,32,Hartford Community College
1,Rory,Stars Hollow,20,Yale
2,Sookie,Stars Hollow,12,Unknown
3,Richard,Hartford,70,Yale


In [117]:
# Test what a standard merge does
df1.merge(df2)

,Name,Occupation,Age,School,Home
0,Lorelai,Manager,32,Hartford Community College,Stars Hollow
1,Rory,Student,20,Yale,Stars Hollow


In [118]:
# Outer merge behaves as expected
df1.merge(df2, how = 'outer')

,Name,Occupation,Age,School,Home
0,Emily,Socialite,60,Smith College,NaN
1,Lorelai,Manager,32,Hartford Community College,Stars Hollow
2,Luke,Owner,40,Stars Hollow High,NaN
3,Richard,NaN,70,Yale,Hartford
4,Rory,Student,20,Yale,Stars Hollow
5,Sookie,NaN,12,Unknown,Stars Hollow


In [119]:
# What if we only specify one column to merge on?
df1.merge(df2, on = 'Name',how = 'outer')

,Name,Occupation,Age_x,School_x,Home,Age_y,School_y
0,Emily,Socialite,60.0,Smith College,NaN,NaN,NaN
1,Lorelai,Manager,32.0,Hartford Community College,Stars Hollow,32.0,Hartford Community College
2,Luke,Owner,40.0,Stars Hollow High,NaN,NaN,NaN
3,Richard,NaN,NaN,NaN,Hartford,70.0,Yale
4,Rory,Student,20.0,Yale,Stars Hollow,20.0,Yale
5,Sookie,NaN,NaN,NaN,Stars Hollow,12.0,Unknown


In [120]:
# Merge on both
df1.merge(df2, on = ['Name','School', 'Age'],how = 'outer')

,Name,Occupation,Age,School,Home
0,Emily,Socialite,60,Smith College,NaN
1,Lorelai,Manager,32,Hartford Community College,Stars Hollow
2,Luke,Owner,40,Stars Hollow High,NaN
3,Richard,NaN,70,Yale,Hartford
4,Rory,Student,20,Yale,Stars Hollow
5,Sookie,NaN,12,Unknown,Stars Hollow


In [121]:
# If one of the columns is actually an index
df2.set_index('Name',inplace=True)
df2

,Home,Age,School
Name,,,
Lorelai,Stars Hollow,32,Hartford Community College
Rory,Stars Hollow,20,Yale
Sookie,Stars Hollow,12,Unknown
Richard,Hartford,70,Yale


In [122]:
# If one of the columns is actually an index
df1.merge(df2, left_on='Name', right_index=True)

,Name,Occupation,Age_x,School_x,Home,Age_y,School_y
0,Lorelai,Manager,32,Hartford Community College,Stars Hollow,32,Hartford Community College
1,Rory,Student,20,Yale,Stars Hollow,20,Yale


In [123]:
df2.reset_index(inplace=True)

---
## 3. Join

`.join()` is a shorthand for index-based merging — both DataFrames must be indexed by the key column.

In [124]:
# Standard join won't work (error!)
df1.join(df2)

ValueError: columns overlap but no suffix specified: Index(['Name', 'Age', 'School'], dtype='str')

In [ ]:
# Join works when we just want to join on the index
df1.set_index('Name',inplace=True)
df2.set_index('Name',inplace=True)

In [ ]:
df1

In [ ]:
df2

In [125]:
# still doesn't work (it doesn't know what to do with the identical columns) 
df1.join(df2)

ValueError: columns overlap but no suffix specified: Index(['Name', 'Age', 'School'], dtype='str')

In [126]:
# We have to define suffixes when we use join (merge did it automatically)
df1.join(df2, lsuffix='_left',rsuffix='_right')

,Name_left,Occupation,Age_left,School_left,Name_right,Home,Age_right,School_right
0,Lorelai,Manager,32,Hartford Community College,Lorelai,Stars Hollow,32,Hartford Community College
1,Rory,Student,20,Yale,Rory,Stars Hollow,20,Yale
2,Luke,Owner,40,Stars Hollow High,Sookie,Stars Hollow,12,Unknown
3,Emily,Socialite,60,Smith College,Richard,Hartford,70,Yale


In [127]:
# Can select different "how" with join
df1.join(df2,how='outer', lsuffix='_left',rsuffix='_right')

,Name_left,Occupation,Age_left,School_left,Name_right,Home,Age_right,School_right
0,Lorelai,Manager,32,Hartford Community College,Lorelai,Stars Hollow,32,Hartford Community College
1,Rory,Student,20,Yale,Rory,Stars Hollow,20,Yale
2,Luke,Owner,40,Stars Hollow High,Sookie,Stars Hollow,12,Unknown
3,Emily,Socialite,60,Smith College,Richard,Hartford,70,Yale


---
## 4. Concatenate

`pd.concat()` stacks DataFrames row-wise (or column-wise) — think of it as appending, not joining on a key.

### 📋 Board Reference

| What | Code | Notes |
|------|------|-------|
| Stack rows | `pd.concat([df1, df2])` | Default: `join='outer'` |
| Keep common cols only | `pd.concat([df1, df2], join='inner')` | |
| Reset index | `pd.concat([df1, df2], ignore_index=True)` | Avoids duplicate indices |
| Stack columns | `pd.concat([df1, df2], axis=1)` | Like cbind — be careful! |
| Add a row | `pd.concat([df, new_row], ignore_index=True)` | Better than deprecated `.append()` |

In [128]:
# First lets reset the indices
df1.reset_index(inplace=True)
df2.reset_index(inplace=True)

In [129]:
df1

,index,Name,Occupation,Age,School
0,0,Lorelai,Manager,32,Hartford Community College
1,1,Rory,Student,20,Yale
2,2,Luke,Owner,40,Stars Hollow High
3,3,Emily,Socialite,60,Smith College


In [132]:
# A standard concatenate
pd.concat([df1,df2])

,index,Name,Occupation,Age,School,Home
0,0,Lorelai,Manager,32,Hartford Community College,NaN
1,1,Rory,Student,20,Yale,NaN
2,2,Luke,Owner,40,Stars Hollow High,NaN
3,3,Emily,Socialite,60,Smith College,NaN
0,0,Lorelai,NaN,32,Hartford Community College,Stars Hollow
1,1,Rory,NaN,20,Yale,Stars Hollow
2,2,Sookie,NaN,12,Unknown,Stars Hollow
3,3,Richard,NaN,70,Yale,Hartford


In [130]:
# Explicitly state how to concatenate (keep all columns that appear in either dataset)
pd.concat([df1,df2],join='outer')

,index,Name,Occupation,Age,School,Home
0,0,Lorelai,Manager,32,Hartford Community College,NaN
1,1,Rory,Student,20,Yale,NaN
2,2,Luke,Owner,40,Stars Hollow High,NaN
3,3,Emily,Socialite,60,Smith College,NaN
0,0,Lorelai,NaN,32,Hartford Community College,Stars Hollow
1,1,Rory,NaN,20,Yale,Stars Hollow
2,2,Sookie,NaN,12,Unknown,Stars Hollow
3,3,Richard,NaN,70,Yale,Hartford


In [131]:
# Only keep common columns
pd.concat([df1,df2],join='inner')

,index,Name,Age,School
0,0,Lorelai,32,Hartford Community College
1,1,Rory,20,Yale
2,2,Luke,40,Stars Hollow High
3,3,Emily,60,Smith College
0,0,Lorelai,32,Hartford Community College
1,1,Rory,20,Yale
2,2,Sookie,12,Unknown
3,3,Richard,70,Yale


In [133]:
# Left/right combinations are not supported
pd.concat([df1,df2],join='left')

ValueError: Only can inner (intersect) or outer (union) join the other axis

In [137]:
# Be careful with indices!
concat_gilmores = pd.concat([df1,df2])
concat_gilmores

,index,Name,Occupation,Age,School,Home
0,0,Lorelai,Manager,32,Hartford Community College,NaN
1,1,Rory,Student,20,Yale,NaN
2,2,Luke,Owner,40,Stars Hollow High,NaN
3,3,Emily,Socialite,60,Smith College,NaN
0,0,Lorelai,NaN,32,Hartford Community College,Stars Hollow
1,1,Rory,NaN,20,Yale,Stars Hollow
2,2,Sookie,NaN,12,Unknown,Stars Hollow
3,3,Richard,NaN,70,Yale,Hartford


In [135]:
# indices might be duplicated
concat_gilmores.loc[2]

,index,Name,Occupation,Age,School,Home
2,2,Luke,Owner,40,Stars Hollow High,NaN
2,2,Sookie,NaN,12,Unknown,Stars Hollow


In [136]:
# You can prevent this by ignoring the index (this will give new indices)
pd.concat([df1,df2],ignore_index=True)

,index,Name,Occupation,Age,School,Home
0,0,Lorelai,Manager,32,Hartford Community College,NaN
1,1,Rory,Student,20,Yale,NaN
2,2,Luke,Owner,40,Stars Hollow High,NaN
3,3,Emily,Socialite,60,Smith College,NaN
4,0,Lorelai,NaN,32,Hartford Community College,Stars Hollow
5,1,Rory,NaN,20,Yale,Stars Hollow
6,2,Sookie,NaN,12,Unknown,Stars Hollow
7,3,Richard,NaN,70,Yale,Hartford


In [138]:
# concatenate the rows (can get very confusing if you aren't careful!)
pd.concat([df1,df2],axis=1)

,index,Name,Occupation,Age,School,index,Name,Home,Age,School
0,0,Lorelai,Manager,32,Hartford Community College,0,Lorelai,Stars Hollow,32,Hartford Community College
1,1,Rory,Student,20,Yale,1,Rory,Stars Hollow,20,Yale
2,2,Luke,Owner,40,Stars Hollow High,2,Sookie,Stars Hollow,12,Unknown
3,3,Emily,Socialite,60,Smith College,3,Richard,Hartford,70,Yale


In [139]:
# Concat is useful for adding rows
new_row = pd.DataFrame([{'Name':'Kurk', 'Home':'Stars Hollow', 'School': 'Unknown'}])
new_row

,Name,Home,School
0,Kurk,Stars Hollow,Unknown


In [140]:
# Concat is useful for adding rows
df2 = pd.concat([df2,new_row],ignore_index=True)
df2

,index,Name,Home,Age,School
0,0.0,Lorelai,Stars Hollow,32.0,Hartford Community College
1,1.0,Rory,Stars Hollow,20.0,Yale
2,2.0,Sookie,Stars Hollow,12.0,Unknown
3,3.0,Richard,Hartford,70.0,Yale
4,NaN,Kurk,Stars Hollow,NaN,Unknown


In [142]:
# Concatenating columns would do an outer
pd.concat([df1,df2],axis=1)

,index,Name,Occupation,Age,School,index,Name,Home,Age,School
0,0.0,Lorelai,Manager,32.0,Hartford Community College,0.0,Lorelai,Stars Hollow,32.0,Hartford Community College
1,1.0,Rory,Student,20.0,Yale,1.0,Rory,Stars Hollow,20.0,Yale
2,2.0,Luke,Owner,40.0,Stars Hollow High,2.0,Sookie,Stars Hollow,12.0,Unknown
3,3.0,Emily,Socialite,60.0,Smith College,3.0,Richard,Hartford,70.0,Yale
4,NaN,NaN,NaN,NaN,NaN,NaN,Kurk,Stars Hollow,NaN,Unknown


In [143]:
# Change it to inner
pd.concat([df1,df2],axis=1,join='inner')

,index,Name,Occupation,Age,School,index,Name,Home,Age,School
0,0,Lorelai,Manager,32,Hartford Community College,0.0,Lorelai,Stars Hollow,32.0,Hartford Community College
1,1,Rory,Student,20,Yale,1.0,Rory,Stars Hollow,20.0,Yale
2,2,Luke,Owner,40,Stars Hollow High,2.0,Sookie,Stars Hollow,12.0,Unknown
3,3,Emily,Socialite,60,Smith College,3.0,Richard,Hartford,70.0,Yale


Let's try reindexing by 'Name' since this is how `axis=1` aligns the dataframe!

In [144]:
df1_idx = df1.set_index('Name')
df2_idx = df2.set_index('Name')

In [145]:
df1_idx

,index,Occupation,Age,School
Name,,,,
Lorelai,0,Manager,32,Hartford Community College
Rory,1,Student,20,Yale
Luke,2,Owner,40,Stars Hollow High
Emily,3,Socialite,60,Smith College


In [147]:
# this is default join='outer'
#    but now we don't get the double Rory and Lorelai because we are using their names

pd.concat([df1_idx, df2_idx], axis=1)

,index,Occupation,Age,School,index,Home,Age,School
Name,,,,,,,,
Lorelai,0.0,Manager,32.0,Hartford Community College,0.0,Stars Hollow,32.0,Hartford Community College
Rory,1.0,Student,20.0,Yale,1.0,Stars Hollow,20.0,Yale
Luke,2.0,Owner,40.0,Stars Hollow High,NaN,NaN,NaN,NaN
Emily,3.0,Socialite,60.0,Smith College,NaN,NaN,NaN,NaN
Sookie,NaN,NaN,NaN,NaN,2.0,Stars Hollow,12.0,Unknown
Richard,NaN,NaN,NaN,NaN,3.0,Hartford,70.0,Yale
Kurk,NaN,NaN,NaN,NaN,NaN,Stars Hollow,NaN,Unknown


---
## 5. 🔬 Activity — Humboldt Student Data

Apply combining operations to real Cal Poly Humboldt institutional data.

---
### Try It 1: Merge Student Datasets 😊

1. **Load the data** — run the cells below to read in the Humboldt student datasets. Check what would happen if you did not include the `skiprows` argument.

2. **Merge** the two DataFrames. Try different `how` arguments — what changes?

3. **Recreate the figure** from the discussion question using the merged dataset.

In [148]:
passing = pd.read_csv('humboldt_data/Humboldt_Passing_Fa23.csv',skiprows=5)

In [149]:
passing.head()

,Year Term,Course Code,Course Title,Enrollment,Number Non-passing
0,2023 - Fall,AHSS101,The Stories We Tell,96,21
1,2023 - Fall,AHSS102,Humboldt Peoples and Places,20,2
2,2023 - Fall,AHSS108,"Nature, Culture, and Food",31,4
3,2023 - Fall,AHSS180,Global Humboldt Seminar,77,5
4,2023 - Fall,AHSS201,"Evol, Creat,and Robot Apocalyp",27,4


In [150]:
first_gen = pd.read_csv('humboldt_data/FirstGenData_Fa23.csv',skiprows=5)

In [151]:
first_gen.head()

,Course,Course Code,FirstGen Enrollment,NotFirstGen Enrollment,FirstGen GPA,NotFirstGen GPA,GPA Gap
0,Theory and Methods,CRGS390,11,12,2.38,3.86,1.48
1,First Year Ere Seminar,ENGR280,10,66,1.70,3.00,1.30
2,Evolution,BIOL307,15,46,2.07,3.02,0.95
3,Organic Chemistry I,CHEM324,13,43,1.69,2.58,0.89
4,"Rock, An American Music",MUS301,10,63,1.93,2.79,0.86


In [159]:
# 2. Merge the two dataframes — try different how arguments

passing.merge(first_gen, left_on=['Course Code', 'Course Title'],right_on = ['Course Code', 'Course'], how='inner').drop(columns = 'Course')


,Year Term,Course Code,Course Title,Enrollment,Number Non-passing,FirstGen Enrollment,NotFirstGen Enrollment,FirstGen GPA,NotFirstGen GPA,GPA Gap
0,2023 - Fall,AHSS101,The Stories We Tell,96,21,16,80,2.23,2.94,0.71
1,2023 - Fall,AHSS180,Global Humboldt Seminar,77,5,13,64,3.36,3.57,0.21
2,2023 - Fall,AHSS480,College Corps,106,1,33,73,3.86,4.00,0.14
3,2023 - Fall,ANTH103,Biological Anthropology,71,14,10,61,2.63,2.80,0.17
4,2023 - Fall,ART108,Beg Graphic Design,49,10,12,37,2.95,2.89,-0.06
...,...,...,...,...,...,...,...,...,...,...
83,2023 - Fall,WLDF365,Ornithology I,48,4,12,36,2.97,2.99,0.02
84,2023 - Fall,WLDF423,Wildlife Mgmt (Nongame Wldf),49,1,14,35,2.72,3.20,0.48
85,2023 - Fall,WLDF495W,Senior Project,38,2,10,28,2.63,3.46,0.83
86,2023 - Fall,ZOOL110,Introductory Zoology,92,23,13,79,1.61,2.21,0.60


In [ ]:
# 3. Recreate the figure from the discussion question


---
## Appendix — Combining Data Quick Reference

### Merge
```python
df1.merge(df2)                                          # inner (default)
df1.merge(df2, how='outer')                             # outer
df1.merge(df2, how='left')                              # left
df1.merge(df2, on='key_col')                            # explicit key
df1.merge(df2, left_on='a', right_on='b')               # different names
df1.merge(df2, left_on='col', right_index=True)         # right df indexed
df1.merge(df2, how='outer', indicator=True)             # debug: see source
```

### Join
```python
# Both dfs must be indexed by the key
df1.set_index('key').join(df2.set_index('key'),
                          lsuffix='_l', rsuffix='_r')
```

### Concat
```python
pd.concat([df1, df2])                       # stack rows (outer join)
pd.concat([df1, df2], join='inner')         # keep only common columns
pd.concat([df1, df2], ignore_index=True)    # reset index
pd.concat([df1, df2], axis=1)               # stack columns
pd.concat([df, new_row], ignore_index=True) # add a row
```

### Checklist flow?

```
Do I want to stack data?
→ YES → concat

NO → Do I want to match rows?

→ YES:
    Match by column? → merge
    Match by index? → join
    Match by time (approx)? → merge_asof```